<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/05_l1c13_training/evaluate_l1c13_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação do experimento de 13 bandas (L1C)
Os modelos treinados com as 13 bandas do L1C (DIVERGENCES D18) no teste L1C, ao lado dos modelos R-G-NIR e das versões publicadas do OmniCloudMask. As contagens dos modelos R-G-NIR e das versões publicadas vêm da avaliação da fase 2 (`notebooks/00_baselines/score_ocm_cloudsen12.ipynb`) e só são lidas aqui.

A comparação do experimento é OCM × Swin, os dois com 13 bandas. A comparação 13 bandas × R-G-NIR de cada arquitetura é só informativa: os modelos R-G-NIR também treinaram no L2A, com o recorte antigo e sem a primeira camada adaptada (LIMITATIONS L2).

In [ ]:
!pip install --quiet "omnicloudmask[legacy]==1.7.0" tacoreader transformers scipy google-auth-oauthlib google-api-python-client

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from IPython.display import display

from cloudband.colab.session import reload_package, start

reload_package("cloudband")

from cloudband.colab.shared_folder import open_shared_folder
from cloudband.pipelines import l1c13_eval
from cloudband.pipelines.phase2_eval import show
from cloudband.train.store import RunStore

In [ ]:
shared = open_shared_folder()

DRIVE_FOLDER = shared.root
RESULTS_DIR = DRIVE_FOLDER / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

session = start(PROJECT_DIR, require_accelerator=True)
store = RunStore(
    results_dir=RESULTS_DIR,
    checkpoint_dir=DRIVE_FOLDER / "checkpoints",
    source_dir=Path(PROJECT_DIR) / "models",
)
store.wake()
if shared.mounted:
    print(f"shared folder: {DRIVE_FOLDER}")
else:
    print(f"shared folder, local copy: {DRIVE_FOLDER} (the last cell sends the results to the Drive)")

In [ ]:
import getpass
import os

import requests

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face token (type 'Read' token, hidden): ").strip()

os.environ.pop("HF_TOKEN", None)
if token:
    reply = requests.get(
        "https://huggingface.co/api/whoami-v2",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if reply.status_code == 200:
        os.environ["HF_TOKEN"] = token
        print(f"token accepted for the account {reply.json().get('name', '?')}")
    else:
        print(f"token refused (HTTP {reply.status_code}); going on without it")
else:
    print("no token: requests use the shared anonymous quota")
del token

In [ ]:
test_table = l1c13_eval.load_test_table()
print("test patches, L1C:", len(test_table))

Cópia local das 13 bandas dos patches de teste (cerca de 7 GB), conferida contra o GDAL.

In [ ]:
cache = l1c13_eval.build_test_cache(Path("/content/cloudband_cache"), test_table)

## Contagens
As seeds de 13 bandas e as da U-Net + RegNetZ-D8 são pontuadas aqui, ou lidas de volta se já foram salvas (`phase2-v2-crop2-counts-test-l1c13-seed{n}.csv`). As dos modelos R-G-NIR e das versões publicadas vêm da fase 2.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

counts = l1c13_eval.score_trained(cache, test_table, store, RESULTS_DIR, DEVICE)
rgn = l1c13_eval.load_rgn_counts(RESULTS_DIR, sorted(counts))
published = l1c13_eval.load_published(RESULTS_DIR)
regnetz = l1c13_eval.score_regnetz(cache, test_table, store, RESULTS_DIR, DEVICE)
combined = l1c13_eval.combine(counts, rgn, published, regnetz)
report = l1c13_eval.build_report(combined)
saved = l1c13_eval.save_report(report, RESULTS_DIR)
print(f"seeds: {sorted(counts)}; {len(saved)} tables saved to {RESULTS_DIR}")

## Onde cada modelo fica
BOA mediano por patch, como no artigo do CloudSEN12 (só patches com a classe e o complemento): as quatro versões publicadas, os modelos R-G-NIR e os de 13 bandas (cada seed e a média).

In [ ]:
l1c13_eval.table_all_models(report).round(2)

In [ ]:
show(l1c13_eval.figure_all_models(report), FIGURES_DIR, l1c13_eval.figure_name("all-models"), display)

## BOA por faixa de pixels da classe
Os patches são separados pela porcentagem dos seus pixels anotados que pertencem à classe do experimento: 0–10%, 10–30%, 30–60% e 60–100% em clear e cloud, e 0–5%, 5–15%, 15–40% e 40–100% em shadow, que ocupa menos do patch. Em cada faixa, o BOA é a mediana do BOA por patch, como no artigo do CloudSEN12: só entram os patches que têm a classe e o complemento dela, e `patches` é quantos são.

Nas barras: OCM publicado 4.0 cada arquitetura com R-G-NIR (tom claro) e com 13 bandas (tom escuro) e a U-Net + RegNetZ-D8 (verde).

In [ ]:
l1c13_eval.table_by_share(combined).round(2)

#### clear: faixas pela porcentagem de pixels de céu limpo

In [ ]:
show(l1c13_eval.figure_by_share(combined, "clear"), FIGURES_DIR, l1c13_eval.figure_name("by-share-clear"), display)

#### cloud/no cloud: faixas pela porcentagem de pixels de nuvem (espessa + fina)

In [ ]:
show(l1c13_eval.figure_by_share(combined, "cloud"), FIGURES_DIR, l1c13_eval.figure_name("by-share-cloud"), display)

#### shadow: faixas pela porcentagem de pixels de sombra de nuvem

In [ ]:
show(l1c13_eval.figure_by_share(combined, "shadow"), FIGURES_DIR, l1c13_eval.figure_name("by-share-shadow"), display)

## Testes estatísticos
### OCM × Swin com 13 bandas (a comparação do experimento)
OCM − Swin, e cada backbone sozinha − Swin, todos com 13 bandas: diferença de BOA por ROI (pontos), intervalo de 95% e p-valor com Holm.

In [ ]:
l1c13_eval.table_ocm_vs_swin(report).round(3)

### 13 bandas × R-G-NIR (informativo)
Cada arquitetura com 13 bandas − a mesma com R-G-NIR, por ROI. Não mede só o efeito das bandas (LIMITATIONS L2).

In [ ]:
l1c13_eval.table_bands(report).round(3)

### U-Net + RegNetZ-D8 × OCM e Swin (sistemas completos)
RegNetZ − OCM e RegNetZ − Swin, todos com 13 bandas, por ROI. A RegNetZ foi treinada com a receita do repositório anterior (DIVERGENCES D19), então a diferença é entre sistemas, não só entre arquiteturas.

In [ ]:
l1c13_eval.table_regnetz(report).round(3)

### 13 bandas × versões publicadas
OCM e Swin com 13 bandas − cada versão publicada, por ROI.

In [ ]:
l1c13_eval.table_vs_published(report).round(3)

## Enviar os resultados para o Drive
Só faz algo quando o Drive não pôde ser montado e o notebook trabalhou numa cópia local: envia para a `cloudband_treino` as contagens, tabelas e figuras geradas.

In [ ]:
sent = shared.push()
print("Drive mounted: results are already there" if shared.mounted else f"{len(sent)} file(s) sent to the Drive")